In [1]:
from markitdown import MarkItDown
from langchain_core.documents import Document
from openai import OpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
from dotenv import load_dotenv
import os
import re
load_dotenv()

True

In [6]:
"test-1".split("_")

['test-1']

In [32]:
client = OpenAI()
md = MarkItDown(llm_client=client, llm_model="gpt-4o")

root = './assets/'
docs = []
for file in os.listdir(root):
    result = md.convert(root + file)
    metadata = file.split("_")
    
    if len(metadata) > 1:
        docs.append({
            "journal": metadata[0],
            "paper_name": metadata[1].split(".")[0],
            "file_name": file,
            "content": result.text_content
        })
    else:
        docs.append({
            "journal": metadata[0].split(".")[0],
            "paper_name": metadata[0].split(".")[0],
            "file_name": file,
            "content": result.text_content
        })

In [33]:
len(docs)

13

# Create Documents Chunk

In [34]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=20,
    length_function=len,
    is_separator_regex=False,
)

In [35]:
chunked_docs = []
for doc in docs:
    meta = {
        "journal": doc['journal'],
        "paper_name": doc['paper_name'],
        "file_name": doc['file_name']
    }
    chunks = text_splitter.create_documents([doc['content']], metadatas=[meta])
    
    for chunk in chunks:
        chunk.page_content += f"\n\n <reference>\nThis chunk refer to:\n Journal={doc['journal']}, Paper Name:{doc['paper_name']}\n</reference>"
    
    chunked_docs += chunks

len(chunked_docs)

1736

In [36]:
chunked_docs[0]

Document(metadata={'journal': 'cureus', 'paper_name': 'The Buccal Pedicle Sliding Flap Technique for Keratinized Tissue Augmentation During the Second-Stage Surgery A Report of Two Cases', 'file_name': 'cureus_The Buccal Pedicle Sliding Flap Technique for Keratinized Tissue Augmentation During the Second-Stage Surgery A Report of Two Cases.pdf'}, page_content='Open Access Case\nReport\n\n DOI: 10.7759/cureus.46362\n\nReview began 09/16/2023\n\nReview ended 09/27/2023\n\nPublished 10/02/2023\n\n© Copyright  2023\n\nAkolu et al. This is an open access article\n\ndistributed under the terms of the Creative\n\nCommons Attribution License CC-BY 4.0.,\n\nwhich permits unrestricted use, distribution,\n\nand reproduction in any medium, provided\n\nthe original author and source are credited.\n\n <reference>\nThis chunk refer to:\n Journal=cureus, Paper Name:The Buccal Pedicle Sliding Flap Technique for Keratinized Tissue Augmentation During the Second-Stage Surgery A Report of Two Cases\n</ref

In [37]:
import faiss
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

index = faiss.IndexFlatL2(len(embeddings.embed_query("START")))
vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

/Users/kuangsmacbook/Library/Caches/pypoetry/virtualenvs/dental-rag-8oW6KoZN-py3.12/lib/python3.12/site-packages/faiss/loader.py:49: DeprecationWarning: numpy.core._multiarray_umath is deprecated and has been renamed to numpy._core._multiarray_umath. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core._multiarray_umath.__cpu_features__.
  from numpy.core._multiarray_umath import __cpu_features__
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
<frozen importlib._bootstr

In [38]:
vector_store.add_documents(documents=chunked_docs)

['173b3ac8-0494-460d-965e-4a38b6736776',
 '0fe8cd4e-bf07-44df-98af-4e2b555f355c',
 '060b31e4-85af-4c08-9810-122eed39aa6a',
 '27d85954-f1ea-44b1-ade6-0da35c8e5212',
 '4d6c4cf3-b590-4f74-8885-24a39c35f571',
 '976ece6b-b826-46cd-8928-aa44e8859287',
 '3930cf48-eaf6-41de-8699-ce5644a67e59',
 '47492e48-2ce8-4c17-bd06-177040cdb81b',
 '1af354f6-97aa-469b-ada8-1b7a01911766',
 '648ea9f7-c777-4509-ba33-34af820b9de1',
 'a339106f-7e87-4448-9877-a9d54cf21161',
 '6ac19a46-02ff-4abe-aca6-382e6a3bbbcc',
 '68c1ea0b-13ce-4e72-a2c0-ef484851fad3',
 'bb2909dc-5b9d-4d1b-91f3-e9b6ce8da097',
 '624f210e-9d06-4d8d-b668-a6ab95bd1068',
 'ee761493-5881-4da7-8f9e-1183efbfd158',
 '1bedb538-777a-4d2c-ae01-16fc5ad29c49',
 'b78ddd4d-4390-41da-a66e-23440a5a8661',
 'f34b232a-880d-4e46-8a02-eacf317f02a4',
 '6272fea4-025f-45f0-858d-af6b86d45963',
 'b60dc9be-7e0c-46f7-be43-43e049915fca',
 '6a8937b3-68cc-4adf-90ab-e26db140388e',
 'a239524f-e590-420b-b3cd-c0146cf113b1',
 '0a468c30-7f22-4096-bd24-d94495575ea5',
 'f2e52a5e-a486-

In [39]:
save_path = "../src/assets/vectorstores"
vector_store.save_local(save_path)

loaded_vector_store = FAISS.load_local(
    save_path, embeddings, allow_dangerous_deserialization=True
)

In [40]:
loaded_vector_store.similarity_search_with_relevance_scores("Torques for immediate implant placement")

[(Document(id='8870091c-fb69-4845-9944-8f8c5f7dccf0', metadata={'journal': 'Nature', 'paper_name': 'Soft and hard tissue changes following immediate implant placement and immediate loading in aesthetic zone a systematic review and meta-analysis', 'file_name': 'Nature_Soft and hard tissue changes following immediate implant placement and immediate loading in aesthetic zone a systematic review and meta-analysis.pdf'}, page_content='Torque values and immediate implant placement\nPrimary stability is a biomechanical aspect to be considered for\nimmediate implant loading61,62. Various experimental studies have\nstated that micromotion of\nimplants should not exceed a\nthreshold of 50 to 150 μm; otherwise, ﬁbrous encapsulation of\nimplants takes place, rather than osseointegration63,64. Hence,\nhigh primary stability is necessary for immediate loading of dental\nimplants; for this reason, modiﬁed drilling protocols in combina-\n\n <reference>\nThis chunk refer to:\n Journal=Nature, Paper Nam